# DLAI Model Merging - Layer-wise and norm ablations

This notebook asks two explanatory questions: (1) which BERT parameter groups cause merge degradation, and (2) is fragility explained by unequal task-vector norms?

Before running, use **Add Input** and attach the saved notebook-02 Output containing `pilot_specialists_seed42.zip`. Select **GPU T4 x2** and enable Internet.

In [ ]:
!nvidia-smi
!find /kaggle/input -maxdepth 3 -type f | head -50

## Install the project and extract seed-42 specialists

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

REPO = 'https://github.com/LeuxLello/Dlai-model-merging.git'
BRANCH = 'codex/multiseed-results'
WORKDIR = Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(WORKDIR)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(WORKDIR)])
sys.path.insert(0, str(WORKDIR / 'src')); os.chdir(WORKDIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

candidates = list(Path('/kaggle/input').rglob('pilot_specialists_seed42.zip'))
assert candidates, 'Attach notebook-02 Output with pilot_specialists_seed42.zip using Add Input.'
SPECIALISTS = Path('/kaggle/working/pilot_specialists_seed42')
if SPECIALISTS.exists(): shutil.rmtree(SPECIALISTS)
with zipfile.ZipFile(candidates[0]) as archive: archive.extractall(SPECIALISTS)
print('Specialist encoders:', list(SPECIALISTS.rglob('encoder.pt')))

## Load states, diagnostics, and evaluators

In [ ]:
import itertools, json, platform
import numpy as np, pandas as pd, torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.ablation import bert_mini_scopes, equal_norm_mean_merge, replace_scope, select_state
from dlai_merge.diagnostics import cosine_similarity, l2_norm, subtract_states
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import mean_merge, ties_merge

assert torch.cuda.is_available(); GPU = torch.cuda.get_device_name(0)
assert torch.cuda.get_device_capability(0)[0] >= 7, 'Use GPU T4 x2, not P100.'
torch.ones(1, device='cuda').add_(1); print('GPU:', GPU)
TASKS = ['sst2', 'imdb', 'mrpc', 'rte']; SEED = 42; BASE = 'prajjwal1/bert-mini'
def artifact(task, name):
    found = list(SPECIALISTS.rglob(f'{task}/seed-{SEED}/{name}')); assert len(found) == 1
    return found[0]
encoders = {t: torch.load(artifact(t, 'encoder.pt'), map_location='cpu', weights_only=True) for t in TASKS}
heads = {t: torch.load(artifact(t, 'head.pt'), map_location='cpu', weights_only=True) for t in TASKS}
base_model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2)
base_encoder = {k: v.detach().cpu().clone() for k, v in base_model.base_model.state_dict().items()}
scopes = bert_mini_scopes(base_encoder.keys())
print({name: len(keys) for name, keys in scopes.items()})
vectors = {t: subtract_states(encoders[t], base_encoder) for t in TASKS}
evaluators = {t: TaskEvaluator(t, heads[t], max_eval_samples=2000, seed=SEED, output_root='/kaggle/working/ablation-eval') for t in TASKS}
references = {t: evaluators[t].evaluate(encoders[t])['primary_score'] for t in TASKS}
references

## Layer-wise diagnostics
We compare one compatible pair (SST-2+IMDb), one fragile pair (IMDb+RTE), and one pair helped by TIES (IMDb+MRPC).

In [ ]:
ABLATION_PAIRS = [('sst2','imdb'), ('imdb','rte'), ('imdb','mrpc')]
diagnostic_rows = []
for left, right in ABLATION_PAIRS:
    for scope, keys in scopes.items():
        lv, rv = select_state(vectors[left], keys), select_state(vectors[right], keys)
        diagnostic_rows.append({
            'pair': f'{left}+{right}', 'task_a': left, 'task_b': right, 'scope': scope,
            'cosine_similarity': cosine_similarity(lv, rv),
            'norm_a': l2_norm(lv), 'norm_b': l2_norm(rv), 'parameter_tensors': len(keys),
        })
layer_diagnostics = pd.DataFrame(diagnostic_rows)
layer_diagnostics

## Layer-wise intervention
For each task, all non-selected groups remain exactly specialist-specific. Only the selected group is replaced by merged parameters. Therefore the measured drop isolates the cost of sharing that group.

In [ ]:
layer_rows = []
for pair in ABLATION_PAIRS:
    states = [encoders[pair[0]], encoders[pair[1]]]
    method_states = {
        'mean': mean_merge(base_encoder, states),
        'ties': ties_merge(base_encoder, states, density=0.2, scale=1.0),
    }
    for method, merged in method_states.items():
        for scope, keys in scopes.items():
            print('Layer ablation:', pair, method, scope)
            for task in pair:
                hybrid = replace_scope(encoders[task], merged, keys)
                score = evaluators[task].evaluate(hybrid)
                layer_rows.append({
                    'pair': '+'.join(pair), 'method': method, 'scope': scope, 'eval_task': task,
                    **score, 'specialist_score': references[task],
                    'retained_ratio': score['primary_score']/references[task],
                    'score_delta': score['primary_score']-references[task],
                })
layer_results = pd.DataFrame(layer_rows)
assert len(layer_results) == 60
layer_summary = (layer_results.groupby(['pair','method','scope'])
    .agg(mean_retained=('retained_ratio','mean'), worst_retained=('retained_ratio','min'))
    .reset_index())
layer_summary

## Equal-norm control
We compare standard Mean merging with a version that rescales both task vectors to their average norm before averaging. This tests whether unequal update magnitude explains fragile pairs.

In [ ]:
norm_rows = []
for pair in itertools.combinations(TASKS, 2):
    variants = {
        'standard_mean': mean_merge(base_encoder, [encoders[pair[0]], encoders[pair[1]]]),
        'equal_norm_mean': equal_norm_mean_merge(base_encoder, encoders[pair[0]], encoders[pair[1]]),
    }
    for variant, merged in variants.items():
        print('Norm control:', pair, variant)
        for task in pair:
            score = evaluators[task].evaluate(merged)
            norm_rows.append({
                'pair': '+'.join(pair), 'variant': variant, 'eval_task': task, **score,
                'specialist_score': references[task],
                'retained_ratio': score['primary_score']/references[task],
                'score_delta': score['primary_score']-references[task],
                'norm_a': l2_norm(vectors[pair[0]]), 'norm_b': l2_norm(vectors[pair[1]]),
            })
norm_results = pd.DataFrame(norm_rows); assert len(norm_results) == 24
norm_summary = (norm_results.groupby(['pair','variant'])
    .agg(mean_retained=('retained_ratio','mean'), worst_retained=('retained_ratio','min'))
    .reset_index())
norm_summary

## Explanatory figures

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
FIGURES = Path('/kaggle/working/ablation_figures'); FIGURES.mkdir(exist_ok=True)
scope_order = ['embeddings','early','late','blocks_all','full']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
layer_plot = layer_summary.copy(); layer_plot['scope'] = pd.Categorical(layer_plot.scope, scope_order, ordered=True); layer_plot = layer_plot.sort_values('scope')
sns.lineplot(data=layer_plot, x='scope', y='mean_retained', hue='pair', style='method', markers=True, dashes=True, ax=axes[0])
axes[0].axhline(1.0,color='gray',linestyle='--'); axes[0].set_title('Where does sharing cause degradation?'); axes[0].tick_params(axis='x',rotation=20)
norm_plot = norm_summary.pivot(index='pair',columns='variant',values='mean_retained').reset_index()
norm_long = norm_plot.melt(id_vars='pair',var_name='variant',value_name='mean_retained')
sns.barplot(data=norm_long,x='pair',y='mean_retained',hue='variant',ax=axes[1])
axes[1].axhline(1.0,color='gray',linestyle='--'); axes[1].set_title('Does equalizing update norms help?'); axes[1].tick_params(axis='x',rotation=30)
fig.tight_layout(); fig.savefig(FIGURES/'layerwise_and_norm_ablation.png',dpi=180,bbox_inches='tight'); plt.show()

## Export compact results

In [ ]:
OUT = Path('/kaggle/working/ablation_results'); OUT.mkdir(exist_ok=True)
layer_diagnostics.to_csv(OUT/'layer_diagnostics.csv',index=False)
layer_results.to_csv(OUT/'layer_results.csv',index=False)
layer_summary.to_csv(OUT/'layer_summary.csv',index=False)
norm_results.to_csv(OUT/'norm_results.csv',index=False)
norm_summary.to_csv(OUT/'norm_summary.csv',index=False)
shutil.copy2(FIGURES/'layerwise_and_norm_ablation.png',OUT/'layerwise_and_norm_ablation.png')
metadata = {
  'purpose':'explanatory layer-wise and equal-norm ablations','seed':SEED,
  'layer_pairs':['sst2+imdb','imdb+rte','imdb+mrpc'],
  'layer_methods':{'mean':{},'ties':{'scale':1.0,'density':0.2}},
  'scopes':{name:list(keys) for name,keys in scopes.items()},
  'gpu':GPU,'python':platform.python_version(),'torch':torch.__version__,
}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
archive=shutil.make_archive('/kaggle/working/layerwise_norm_ablation_results','zip',OUT)
print(archive); print(*sorted(str(p) for p in OUT.iterdir()),sep='\n')